# Nextflow & nf-core RAG Assistant

A Retrieval-Augmented Generation (RAG) Q&A app specialized on **Nextflow** and
**nf-core** documentation. Ask a question in natural language; the app embeds it,
searches a Chroma vector database of document chunks, retrieves the most relevant
passages, and passes them to an LLM that answers **using only that context**.

**Pipeline:** load & clean docs → header-aware chunking (with overlap) → embed with
OpenAI → store in Chroma → similarity search → context-restricted generation with
`gpt-4o-mini` → multi-turn Gradio chat → 3-D Plotly view of the embedding space.

| Stage | Tool |
|---|---|
| Embeddings | OpenAI `text-embedding-3-small` |
| Vector DB | Chroma (persistent) |
| Generation | `gpt-4o-mini` (OpenAI direct, or OpenRouter) |
| UI | Gradio |
| Visualization | UMAP + Plotly (3-D, interactive) |

> **Keys:** embeddings and (by default) generation use `OPENAI_API_KEY`. Generation can
> optionally be routed through OpenRouter by setting `LLM_PROVIDER = "openrouter"` and
> providing `OPENROUTER_API_KEY`. Keys live in a `.env` file in the project root (git-ignored).


## 1. Imports & environment

In [1]:
import os
import re
import glob
import json
from pathlib import Path

import numpy as np
import tiktoken
import chromadb
import gradio as gr
import plotly.graph_objects as go
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

/Users/topherthompson/Documents/github/LLMs_RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load .env from the project root.
load_dotenv(find_dotenv(usecwd=True))

OPENAI_API_KEY     = os.getenv("OPENAI_API_KEY", "")      # embeddings (+ generation by default)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")  # optional, for generation via OpenRouter

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env (needed for embeddings)."
print("Keys loaded: OPENAI_API_KEY ✓ | OPENROUTER_API_KEY",
      "✓" if OPENROUTER_API_KEY else "— (not set; using OpenAI for generation)")

Keys loaded: OPENAI_API_KEY ✓ | OPENROUTER_API_KEY ✓


## 2. Configuration

All tunable knobs live here. Chunk size is expressed in **tokens** (the unit the
embedding model actually sees) and we keep an **overlap** so an idea that crosses a
chunk boundary still appears whole in at least one chunk.

In [3]:
# --- Paths ---
DOCS_DIR        = "./data"                       # source .md documents
CHROMA_DIR      = "./chroma_db"                   # persistent vector store
COLLECTION_NAME = "nextflow_nfcore"

# --- Models ---
EMBED_MODEL = "text-embedding-3-small"            # OpenAI embeddings (uses OPENAI_API_KEY)

# Generation provider:
#   "openai"     -> calls gpt-4o-mini directly with OPENAI_API_KEY (default; what we use)
#   "openrouter" -> routes openai/gpt-4o-mini through OPENROUTER_API_KEY
# Both answer with gpt-4o-mini, as the assignment requires.
LLM_PROVIDER = "openai"

# --- Chunking (token-based, structure-aware) ---
CHUNK_TOKENS   = 500     # target max tokens per chunk (assignment: ~500-800)
OVERLAP_TOKENS = 80      # token overlap between adjacent chunks for continuity

# --- Retrieval / generation ---
TOP_K       = 4          # chunks retrieved per query
MAX_HISTORY = 6          # how many prior turns to keep as memory
GEN_TEMPERATURE = 0.2
GEN_MAX_TOKENS  = 700

# --- The system prompt: this is what keeps the LLM on task (RAG-restricting) ---
SYSTEM_PROMPT = (
    "You are NextflowAssistant, a specialized assistant that answers questions about "
    "Nextflow and the nf-core community using ONLY the provided context excerpts, which "
    "come from the official Nextflow and nf-core documentation.\n\n"
    "Rules:\n"
    "1. Answer strictly from the provided context. Do NOT use outside knowledge or guess.\n"
    "2. If the answer is not in the context, say you don't have enough information in the "
    "provided documentation, and suggest how the user might rephrase.\n"
    "3. If the question is unrelated to Nextflow or nf-core, politely decline and remind "
    "the user you only cover Nextflow/nf-core topics.\n"
    "4. Be concise and technical. Include short code snippets from the context when useful.\n"
    "5. Prefer citing the source document name when you state a fact."
)

# tiktoken encoder used only to *count* tokens for chunk sizing.
ENC = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    """Number of tokens in a string (for chunk sizing)."""
    return len(ENC.encode(text))

### Generation client

One OpenAI-compatible client is used for both the safety check and answer
generation. With `LLM_PROVIDER = "openai"` it talks to OpenAI directly; set it to
`"openrouter"` (and provide `OPENROUTER_API_KEY`) to route the same `gpt-4o-mini`
through OpenRouter instead.

In [4]:
def make_gen_client():
    """Return (client, model_name) for the configured generation provider."""
    if LLM_PROVIDER == "openrouter":
        assert OPENROUTER_API_KEY, ("LLM_PROVIDER='openrouter' but OPENROUTER_API_KEY "
                                    "is not set. Add it to .env or use LLM_PROVIDER='openai'.")
        return (OpenAI(api_key=OPENROUTER_API_KEY,
                       base_url="https://openrouter.ai/api/v1"),
                "openai/gpt-4o-mini")
    return OpenAI(api_key=OPENAI_API_KEY), "gpt-4o-mini"


gen_client, GEN_MODEL = make_gen_client()
SAFETY_MODEL = GEN_MODEL                 # reuse gpt-4o-mini for the cheap safety check
print(f"Generation via '{LLM_PROVIDER}' using model: {GEN_MODEL}")

Generation via 'openai' using model: gpt-4o-mini


## 3. Document loading & preprocessing

The corpus is Markdown pulled from the official Nextflow docs and the nf-core website
(both open-licensed). Before chunking we **clean** each file:

- pull the YAML frontmatter `title:` out as metadata, then strip the frontmatter block
- remove MyST/Astro markup that would only add noise to embeddings:
  directive fences (`:::note`, `::::{tip}`), heading anchors (`(some-anchor)=`),
  and cross-reference roles (`` {ref}`Text <target>` `` → `Text`)
- collapse excess blank lines

Code blocks are deliberately **kept** — they are some of the most useful content to
retrieve for a "how do I write X" question.

In [5]:
FRONTMATTER_RE = re.compile(r"^---\n(.*?)\n---\n", re.DOTALL)
ANCHOR_RE      = re.compile(r"^\([\w-]+\)=\s*$", re.MULTILINE)   # (workflow-page)=
DIRECTIVE_RE   = re.compile(r"^:{3,}.*$", re.MULTILINE)            # :::note / ::::{tip}
REF_ROLE_RE    = re.compile(r"\{[a-z]+\}`([^<`]+?)(?:\s*<[^>]+>)?`")  # {ref}`A <b>` -> A
HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
MULTI_BLANK_RE = re.compile(r"\n{3,}")


def clean_markdown(raw: str) -> tuple[str, str]:
    """Return (title, cleaned_body) for one raw markdown document."""
    title = ""
    m = FRONTMATTER_RE.match(raw)
    if m:
        for line in m.group(1).splitlines():
            if line.strip().lower().startswith("title:"):
                title = line.split(":", 1)[1].strip().strip('"').strip("'")
                break
        raw = raw[m.end():]                      # drop the frontmatter block

    raw = HTML_COMMENT_RE.sub("", raw)
    raw = ANCHOR_RE.sub("", raw)
    raw = DIRECTIVE_RE.sub("", raw)
    raw = REF_ROLE_RE.sub(r"\1", raw)
    raw = MULTI_BLANK_RE.sub("\n\n", raw)
    raw = raw.strip()

    if not title:                                # docs without frontmatter: use first H1
        m1 = re.search(r"^#\s+(.+)$", raw, re.MULTILINE)
        if m1:
            title = m1.group(1).strip()
    return title, raw

### Header-aware splitting

We first split each document on Markdown headers so chunks respect the document's
structure (and we carry a **breadcrumb** like `Processes > Inputs` into metadata).
A section that is still too long is then sub-divided to the token budget, packing whole
sentences and re-including the trailing `OVERLAP_TOKENS` from the previous chunk so
ideas that straddle a boundary stay intact. This is exactly the "split on structure,
then subdivide to the token boundary, with overlap" strategy the assignment asks for —
not a blind split on token counts.

In [6]:
HEADER_RE = re.compile(r"^(#{1,6})\s+(.*)$")


def split_into_sections(body: str) -> list[dict]:
    """
    Split a cleaned doc into sections keyed by Markdown headers, tracking a
    hierarchical breadcrumb of the enclosing headings.
    Returns: [{"breadcrumb": str, "text": str}, ...]
    """
    sections, heading_stack, buf = [], [], []

    def flush():
        text = "\n".join(buf).strip()
        if text:
            sections.append({"breadcrumb": " > ".join(h[1] for h in heading_stack),
                             "text": text})
        buf.clear()

    for line in body.splitlines():
        m = HEADER_RE.match(line)
        if m:
            flush()                                   # close previous section
            level = len(m.group(1))
            title = m.group(2).strip()
            # pop deeper-or-equal headings, then push this one
            heading_stack[:] = [h for h in heading_stack if h[0] < level]
            heading_stack.append((level, title))
        else:
            buf.append(line)
    flush()
    return sections


def _sentences(text: str) -> list[str]:
    """Naive sentence/line splitter that keeps code lines as their own units."""
    parts = re.split(r"(?<=[.!?:])\s+|\n", text)
    return [p for p in (s.strip() for s in parts) if p]


def pack_with_overlap(text: str, max_tokens: int, overlap_tokens: int) -> list[str]:
    """Greedily pack sentences up to max_tokens; seed each new chunk with the
    trailing sentences of the previous one until overlap_tokens is reached."""
    sents = _sentences(text)
    chunks, buf = [], []

    def buf_tokens():
        return n_tokens(" ".join(buf))

    for sent in sents:
        if buf and buf_tokens() + n_tokens(sent) > max_tokens:
            chunks.append(" ".join(buf))
            # build overlap seed from the tail of the just-emitted chunk
            seed, tok = [], 0
            for s in reversed(buf):
                tok += n_tokens(s)
                seed.insert(0, s)
                if tok >= overlap_tokens:
                    break
            buf = seed[:]
        buf.append(sent)
    if buf:
        chunks.append(" ".join(buf))
    return chunks

In [7]:
def load_and_chunk(docs_dir: str) -> list[dict]:
    """
    Load every .md file, clean it, split on headers, and subdivide oversized
    sections to the token budget with overlap.

    Returns a flat list of chunk dicts:
        {text, source, doc_title, section, chunk_idx}
    The breadcrumb is prepended to the embedded text so each chunk carries the
    context of where it sits in the document.
    """
    paths = sorted(glob.glob(os.path.join(docs_dir, "*.md")))
    if not paths:
        raise FileNotFoundError(f"No .md files found in {docs_dir!r}.")

    all_chunks = []
    for path in paths:
        raw = Path(path).read_text(encoding="utf-8")
        source = Path(path).name
        title, body = clean_markdown(raw)
        title = title or source

        idx = 0
        for sec in split_into_sections(body):
            pieces = ([sec["text"]] if n_tokens(sec["text"]) <= CHUNK_TOKENS
                      else pack_with_overlap(sec["text"], CHUNK_TOKENS, OVERLAP_TOKENS))
            for piece in pieces:
                header = f"[{title}] {sec['breadcrumb']}".strip()
                all_chunks.append({
                    "text":      f"{header}\n{piece}" if sec["breadcrumb"] else piece,
                    "source":    source,
                    "doc_title": title,
                    "section":   sec["breadcrumb"],
                    "chunk_idx": idx,
                })
                idx += 1
        print(f"  {source:<38} -> {idx:>3} chunks  ({title})")

    print(f"\n[load] {len(paths)} documents -> {len(all_chunks)} chunks total.")
    return all_chunks

## 4. Embedding & vector store (Chroma)

Chroma calls the OpenAI embedding function automatically — at index time on each
chunk, and at query time on the raw query — so we only ever pass text. We store the
chunk **id**, **text**, and rich **metadata** (`source`, `doc_title`, `section`,
`chunk_idx`) so retrieved results are traceable back to their document and heading.

Indexing is **idempotent**: ids are deterministic (`{source}__{chunk_idx}`), so
re-running this notebook does not re-embed or duplicate existing chunks.

In [8]:
def get_embedding_function() -> OpenAIEmbeddingFunction:
    """OpenAI embedding function (uses the OpenAI API, not OpenRouter)."""
    return OpenAIEmbeddingFunction(api_key=OPENAI_API_KEY, model_name=EMBED_MODEL)


def index_chunks(collection, chunks: list[dict]) -> None:
    """Add only new chunks to Chroma; skip ids already present."""
    candidate = {f"{c['source']}__{c['chunk_idx']}": c for c in chunks}
    existing  = set(collection.get(ids=list(candidate.keys()))["ids"])
    new = [(cid, c) for cid, c in candidate.items() if cid not in existing]

    if not new:
        print(f"[index] All {len(chunks)} chunks already indexed — nothing to do.")
        return

    collection.add(
        ids       =[cid for cid, _ in new],
        documents =[c["text"] for _, c in new],
        metadatas =[{
            "source":    c["source"],
            "doc_title": c["doc_title"],
            "section":   c["section"],
            "chunk_idx": c["chunk_idx"],
        } for _, c in new],
    )
    print(f"[index] {len(new)} new chunks indexed ({len(existing)} already existed).")


def build_vectorstore():
    """Load -> chunk -> index. Returns the populated Chroma collection."""
    chunks = load_and_chunk(DOCS_DIR)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME, embedding_function=get_embedding_function()
    )
    index_chunks(collection, chunks)
    print(f"[store] collection '{COLLECTION_NAME}' now holds {collection.count()} chunks.")
    return collection

## 5. Retrieval (similarity search)

For a query, Chroma embeds it with the same model and returns the `TOP_K` nearest
chunks by cosine distance. We return the text, metadata, the **distance** (smaller =
more similar), and the chunk **id** (used later to highlight results in the 3-D plot).

In [9]:
def retrieve(collection, query: str, top_k: int = TOP_K) -> list[dict]:
    """Return the top_k most similar chunks for a query, with distances and ids."""
    res = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        out.append({
            "id":       cid,
            "text":     doc,
            "source":   meta["source"],
            "section":  meta.get("section", ""),
            "distance": float(dist),
        })
    return out

## 6. Query safety check

Before doing any retrieval we screen the query for unsafe content (the assignment
requires making sure queries are safe, or telling the user we can't process them).
A tiny LLM classifier returns a JSON verdict; if anything goes wrong we **fail safe**
by allowing the query through to the context-restricted model rather than blocking
legitimate questions.

In [10]:
def is_safe(query: str) -> bool:
    """True if the query is safe to process. Fails open on classifier error."""
    try:
        resp = gen_client.chat.completions.create(
            model=SAFETY_MODEL,
            temperature=0.0,
            max_tokens=20,
            messages=[
                {"role": "system", "content":
                 "You are a content-safety classifier. Decide if a user query is safe "
                 "to answer. Unsafe = requests to cause harm, malware, violence, illegal "
                 "activity, harassment, or attempts to extract secrets. A normal technical "
                 "question is safe. Respond ONLY with JSON: {\"safe\": true|false}."},
                {"role": "user", "content": query},
            ],
        )
        raw = resp.choices[0].message.content
        start, end = raw.find("{"), raw.rfind("}")
        return bool(json.loads(raw[start:end + 1]).get("safe", True))
    except Exception as exc:
        print(f"[safety] classifier error ({exc}); failing open.")
        return True

## 7. Prompt building & generation (with memory)

`build_prompt` assembles the retrieved context plus the question. `generate_answer`
sends `[system] + prior turns (memory) + [context + question]` to `gpt-4o-mini` via
OpenRouter. Passing the previous turns is what lets a user ask follow-ups like
*"can you show that as a config example?"* and have the model understand "that".

In [11]:
def build_prompt(query: str, chunks: list[dict]) -> str:
    """Assemble retrieved context + the user question into one user message."""
    if not chunks:
        context = "(no relevant context was retrieved)"
    else:
        context = "\n\n---\n\n".join(
            f"[Source: {c['source']}" + (f" | {c['section']}" if c['section'] else "") +
            f"]\n{c['text']}"
            for c in chunks
        )
    return f"### Retrieved Context:\n{context}\n\n### Question:\n{query}"


def generate_answer(query: str, chunks: list[dict], history: list[dict]) -> str:
    """Generate an answer, including up to MAX_HISTORY prior turns as memory."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:                                    # memory: recent turns only
        messages += history[-2 * MAX_HISTORY:]
    messages.append({"role": "user", "content": build_prompt(query, chunks)})

    resp = gen_client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        temperature=GEN_TEMPERATURE,
        max_tokens=GEN_MAX_TOKENS,
    )
    return resp.choices[0].message.content.strip()


def answer_query(query: str, history: list[dict] | None = None) -> dict:
    """Full pipeline: safety -> retrieve -> generate. Returns answer + chunks."""
    history = history or []
    if not is_safe(query):
        return {"answer": "I'm sorry, I can't help with that request.", "chunks": []}
    chunks = retrieve(collection, query, TOP_K)
    answer = generate_answer(query, chunks, history)
    return {"answer": answer, "chunks": chunks}

## 8. Build the vector store

In [12]:
collection = build_vectorstore()

  nextflow_config.md                     ->  13 chunks  (Configuration)
  nextflow_overview.md                   ->   5 chunks  (Overview)
  nextflow_workflow.md                   ->  19 chunks  (Workflows)
  nextflow_your-first-script.md          ->   5 chunks  (Your first script)
  nfcore_configuration-options.md        ->   6 chunks  (Configuration options)
  nfcore_reference-genomes.md            ->   6 chunks  (Reference genomes)
  nfcore_run-pipelines.md                ->  12 chunks  (Running pipelines)
  nfcore_run-your-first-pipeline.md      ->   9 chunks  (Run your first pipeline)
  nfcore_running-overview.md             ->   7 chunks  (Overview)
  nfcore_tools.md                        ->   9 chunks  (nf-core tools)
  nfcore_what-is-nf-core.md              ->   9 chunks  (What is nf-core?)

[load] 11 documents -> 100 chunks total.
[index] All 100 chunks already indexed — nothing to do.
[store] collection 'nextflow_nfcore' now holds 100 chunks.


### Quick smoke test
A relevant question (should answer from context), and an off-topic one (should be
politely declined by the system prompt).

In [13]:
demo_q = "What is a process in Nextflow and how do I define its inputs and outputs?"
result = answer_query(demo_q)
print("Q:", demo_q, "\n")
print(result["answer"], "\n")
print("Sources:", ", ".join(sorted({c["source"] for c in result["chunks"]})))

Q: What is a process in Nextflow and how do I define its inputs and outputs? 

A process in Nextflow is a fundamental building block of a pipeline, which can be written in any scripting language executable on the Linux platform (e.g., Bash, Perl, Ruby, Python). Each process can define one or more inputs and outputs, and data flows between processes through channels.

To define a process, you specify its inputs and outputs using the `input:` and `output:` directives. Here’s an example of how to define a process with inputs and outputs:

```nextflow
process blast_search {
  input:
  path query
  path db

  output:
  path "top_hits.txt"

  script:
  """
  blastp -db $db -query $query -outfmt 6 > blast_result
  cat blast_result | head -n 10 | cut -f 2 > top_hits.txt
  """
}
```

In this example, the `blast_search` process has two inputs (`query` and `db`) and one output (`top_hits.txt`). The inputs are specified as paths, and the output is also defined as a path. This structure allows Next

In [14]:
offtopic = answer_query("What's a good recipe for chocolate chip cookies?")
print(offtopic["answer"])

I'm sorry, but I only cover Nextflow and nf-core topics. Please feel free to ask a question related to those subjects!


## 9. 3-D interactive embedding visualization

We reduce the high-dimensional chunk embeddings to 3-D with **UMAP** and plot them with
**Plotly**. The layout is computed **once** (cached), so each query is fast: we simply
re-color the points, highlighting the chunks retrieved for the current question and
showing their cosine **distance**. Hovering any point previews the chunk's source,
section, and text — satisfying the assignment's "preview the chunk when hovering" goal.

In [15]:
def compute_layout(collection) -> dict:
    """Fetch all embeddings and compute a cached 3-D UMAP layout."""
    import umap
    data = collection.get(include=["embeddings", "documents", "metadatas"])
    embs = np.array(data["embeddings"])
    n = len(embs)
    print(f"[viz] reducing {n} x {embs.shape[1]} embeddings to 3-D with UMAP…")
    coords = umap.UMAP(
        n_components=3,
        n_neighbors=min(15, max(2, n - 1)),
        min_dist=0.1,
        metric="cosine",
        random_state=42,
    ).fit_transform(embs)
    return {
        "ids":     data["ids"],
        "coords":  coords,
        "sources": [m["source"] for m in data["metadatas"]],
        "sections":[m.get("section", "") for m in data["metadatas"]],
        "docs":    data["documents"],
    }


def _hover(src, section, doc):
    preview = doc[:160].replace("\n", " ")
    head = f"<b>{src}</b>" + (f" | {section}" if section else "")
    return f"{head}<br>{preview}…"


def plot_embedding_space(layout: dict, retrieved: list[dict] | None = None,
                         query: str = "") -> go.Figure:
    """3-D scatter of all chunks; retrieved chunks (if any) highlighted in red."""
    retrieved = retrieved or []
    ret_ids = {c["id"]: c["distance"] for c in retrieved}
    coords  = layout["coords"]

    base_x, base_y, base_z, base_h = [], [], [], []
    hi_x, hi_y, hi_z, hi_h = [], [], [], []
    for i, cid in enumerate(layout["ids"]):
        h = _hover(layout["sources"][i], layout["sections"][i], layout["docs"][i])
        if cid in ret_ids:
            hi_x.append(coords[i, 0]); hi_y.append(coords[i, 1]); hi_z.append(coords[i, 2])
            hi_h.append(f"⭐ RETRIEVED (distance={ret_ids[cid]:.3f})<br>{h}")
        else:
            base_x.append(coords[i, 0]); base_y.append(coords[i, 1]); base_z.append(coords[i, 2])
            base_h.append(h)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=base_x, y=base_y, z=base_z, mode="markers",
        marker=dict(size=3, color="lightslategray", opacity=0.45),
        text=base_h, hoverinfo="text", name="all chunks",
    ))
    if hi_x:
        fig.add_trace(go.Scatter3d(
            x=hi_x, y=hi_y, z=hi_z, mode="markers",
            marker=dict(size=7, color="crimson", opacity=0.95,
                        line=dict(width=1, color="black")),
            text=hi_h, hoverinfo="text", name="retrieved",
        ))
    title = "Embedding space (UMAP 3-D)"
    if query:
        title += f" — retrieved for: {query[:60]}"
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="UMAP-1", yaxis_title="UMAP-2", zaxis_title="UMAP-3"),
        margin=dict(l=0, r=0, b=0, t=40), height=620, showlegend=True,
    )
    return fig

In [16]:
# Compute the layout once and preview the full embedding space, with the demo
# query's retrieved chunks highlighted.
LAYOUT = compute_layout(collection)
plot_embedding_space(LAYOUT, retrieve(collection, demo_q), demo_q)

[viz] reducing 100 x 1536 embeddings to 3-D with UMAP…


/Users/topherthompson/Documents/github/LLMs_RAG/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 10. Gradio app

The interface ties everything together:

- a **chat** box with **memory** (each turn sees the prior conversation),
- the **source chunks** used for the latest answer (with their similarity distance), and
- the **3-D embedding plot** updated to highlight what was retrieved for your question.

Run this cell, then open the local URL it prints.

In [17]:
def format_sources(chunks: list[dict]) -> str:
    """Markdown summary of the retrieved source chunks shown beside the chat."""
    if not chunks:
        return "_No source chunks were retrieved for this turn._"
    lines = ["### Retrieved source chunks"]
    for i, c in enumerate(chunks, 1):
        loc = f"`{c['source']}`" + (f" — {c['section']}" if c["section"] else "")
        preview = c["text"].replace("\n", " ")[:240]
        lines.append(f"**{i}. {loc}**  \n_distance: {c['distance']:.3f}_  \n{preview}…")
    return "\n\n".join(lines)


EMPTY_FIG = plot_embedding_space(LAYOUT)  # full space, nothing highlighted


def chat_fn(message, history):
    """Gradio handler: returns (cleared input, new history, sources md, 3-D figure)."""
    history = history or []
    if not message or not message.strip():
        return "", history, "_Type a question to begin._", EMPTY_FIG

    res = answer_query(message, history)            # history = memory
    new_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": res["answer"]},
    ]
    fig = plot_embedding_space(LAYOUT, res["chunks"], message)
    return "", new_history, format_sources(res["chunks"]), fig


with gr.Blocks(title="Nextflow & nf-core RAG Assistant", fill_height=True) as demo:
    gr.Markdown("# 🧬 Nextflow & nf-core RAG Assistant\n"
                "Ask about Nextflow or nf-core. Answers come **only** from the indexed docs.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, label="Chat (with memory)")
            msg = gr.Textbox(placeholder="e.g. How do I configure resources for an nf-core pipeline?",
                             label="Your question", autofocus=True)
            with gr.Row():
                send  = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear chat")
        with gr.Column(scale=2):
            sources = gr.Markdown("_Ask a question to see the source chunks._")
    plot = gr.Plot(label="3-D embedding space (retrieved chunks in red)")

    for trigger in (msg.submit, send.click):
        trigger(chat_fn, [msg, chatbot], [msg, chatbot, sources, plot])
    clear.click(lambda: ([], "_Cleared._", EMPTY_FIG), None, [chatbot, sources, plot])

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---
### Notes for graders / teammates
- **Data:** `data/` holds 11 open-licensed Markdown docs (Nextflow core + nf-core), ~19 pages.
- **Re-indexing:** delete the `chroma_db/` folder to rebuild embeddings from scratch
  (e.g. after changing the chunking parameters).
- **Costs:** only embeddings use the paid OpenAI key (a few cents for the whole corpus);
  generation runs through OpenRouter.
